# Composite stress tests

Stack **rates**, **credit**, and **FX** using **`compose_scenarios`**, inspect **historical replay** templates, and see how **priority** orders specs when merging.

## Concept

- **Composition**: `compose_scenarios` accepts a JSON **list** of `ScenarioSpec` objects and returns a **single** spec whose operations are merged per engine rules.
- **Authoring**: each per-factor spec is built from typed **`OperationSpec`** operations — `curve_parallel_bp` with `CurveKind.discount()` / `CurveKind.par_cds()` for rates and credit, `market_fx_pct` for FX.
- **Priority**: each spec carries an integer **`priority`** (lower runs earlier in composition—see merged ordering in printed JSON).
- **Historical templates**: built-ins such as **`gfc_2008`**, **`covid_2020`**, **`rate_shock_2022`** bundle multiple risk factors; **`list_template_components`** exposes rates/credit/vol/FX building blocks.
- **FX state**: the pair must exist in the typed `FxMatrix` before the market is serialized; scenario shocks then preserve the same pair and policy state.


### Composed vs sequential application

**`compose_scenarios`** merges multiple `ScenarioSpec` values into **one** spec. Each input spec has an integer **`priority`**; the merged **`operations`** list reflects the engine’s ordering (**lower priority number ⇒ earlier in the merged stack**—print the numbered operations below to verify).

**Sequential** stress means calling **`apply_scenario_to_market`** on the **base** market, then again on the returned **`market_json`**, and so on—the timeline is whatever order you choose in code. **Composed** stress applies **all** merged operations in **one** call. Because discounting, credit, and FX steps **do not always commute**, the **terminal market** from “compose once” vs “apply three times” can differ; comparing the two is a useful sanity check.

In [1]:
import json

from finstack_quant.scenarios import (
    CurveKind,
    OperationSpec,
    apply_scenario_to_market,
    build_scenario_spec,
    build_template_component,
    compose_scenarios,
    list_builtin_template_metadata,
    list_builtin_templates,
    list_template_components,
    validate_scenario_spec,
)
print("apply_scenario_to_market:", callable(apply_scenario_to_market), "build_template_component:", callable(build_template_component))

print("Historical / stress templates:", list_builtin_templates())
meta = list_builtin_template_metadata()
print("Metadata JSON chars:", len(meta))

for tid in ("gfc_2008", "covid_2020", "rate_shock_2022"):
    if tid in list_builtin_templates():
        comps = list_template_components(tid)
        print(f"Template {tid} components:", comps)


def ops_json(*ops: OperationSpec) -> str:
    """Serialize typed operations into the JSON list `build_scenario_spec` expects."""
    return json.dumps([json.loads(op.to_json()) for op in ops])


DISCOUNT = CurveKind.discount()
PAR_CDS = CurveKind.par_cds()

# One spec per risk factor, each authored with the typed builders and tagged with a
# composition priority (lower runs earlier in the merged stack).
rates_json = build_scenario_spec(
    "step_rates",
    ops_json(OperationSpec.curve_parallel_bp(DISCOUNT, "USD-OIS", 40.0)),
    "Rates +40bp",
    "",
    priority=10,
)
credit_json = build_scenario_spec(
    "step_credit",
    ops_json(OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-HAZARD", 70.0, "USD-OIS")),
    "Credit +70bp",
    "",
    priority=20,
)
fx_json = build_scenario_spec(
    "step_fx",
    ops_json(OperationSpec.market_fx_pct("EUR", "USD", 3.0)),
    "FX +3% EURUSD",
    "",
    priority=5,
)

specs_list = [json.loads(rates_json), json.loads(credit_json), json.loads(fx_json)]
composed = compose_scenarios(json.dumps(specs_list))
comp_obj = json.loads(composed)
print("Composed scenario id:", comp_obj.get("id"))
print("Composed operations count:", len(comp_obj.get("operations", [])))
print("validate composed:", validate_scenario_spec(composed))
print("First 500 chars of composed:", composed[:500])
print("Composed operations (verify merge / priority ordering):")
for i, op in enumerate(comp_obj.get("operations", [])):
    tag = op.get("curve_id") or op.get("base") or op.get("curve_kind")
    print(f"  [{i}] kind={op.get('kind')!r} ref={tag!r}")


apply_scenario_to_market: True build_template_component: True
Historical / stress templates: ['gfc_2008', 'covid_2020', 'rate_shock_2022', 'svb_2023', 'ltcm_1998']
Metadata JSON chars: 3717
Template gfc_2008 components: ['gfc_2008_rates', 'gfc_2008_credit', 'gfc_2008_equity', 'gfc_2008_vol', 'gfc_2008_fx']
Template covid_2020 components: ['covid_2020_rates', 'covid_2020_credit', 'covid_2020_equity', 'covid_2020_vol', 'covid_2020_fx']
Template rate_shock_2022 components: ['rate_shock_2022_rates', 'rate_shock_2022_credit', 'rate_shock_2022_equity', 'rate_shock_2022_vol', 'rate_shock_2022_fx']
Composed scenario id: step_fx+step_rates+step_credit
Composed operations count: 3
validate composed: True
First 500 chars of composed: {"id":"step_fx+step_rates+step_credit","name":"FX +3% EURUSD + Rates +40bp + Credit +70bp","description":null,"operations":[{"kind":"market_fx_pct","base":"EUR","quote":"USD","pct":3.0},{"kind":"curve_parallel_bp","curve_kind":"discount","curve_id":"USD-OIS","bp":40.

In [2]:
import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF, build_demo_market
from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import MarketContext

bd = DEMO_AS_OF
AS_OF = str(bd)

# The shared demo market already carries everything this stress needs: a USD-OIS
# discount curve, the CORP-HAZARD credit curve (with par spreads), and a typed
# `FxMatrix` holding EUR/USD. Note the FX pair must be present *before* the market
# is serialized, otherwise `market_fx_pct` has nothing to shock.
mc = build_demo_market(bd)
market_json = mc.to_json()
base_eurusd = mc.fx.rate(Currency("EUR"), Currency("USD"), bd).rate
print(f"Market with typed FX quote; EUR/USD={base_eurusd}")

# Multi-step composition with explicit priorities: FX 5, rates 10, credit 20
# (lower number runs earlier in the merged stack).
rates = build_scenario_spec(
    "r",
    ops_json(OperationSpec.curve_parallel_bp(DISCOUNT, "USD-OIS", 35.0)),
    "Rates +35bp",
    "",
    10,
)
credit = build_scenario_spec(
    "c",
    ops_json(OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-HAZARD", 60.0, "USD-OIS")),
    "Credit +60bp",
    "",
    20,
)
fx = build_scenario_spec(
    "f",
    ops_json(OperationSpec.market_fx_pct("EUR", "USD", 4.0)),
    "FX +4% EURUSD",
    "",
    5,
)
composed = compose_scenarios(json.dumps([json.loads(rates), json.loads(credit), json.loads(fx)]))
comp2 = json.loads(composed)
print("Composed spec priorities embedded; merged id:", comp2["id"])
print("Merged operation stack:")
for i, op in enumerate(comp2.get("operations", [])):
    tag = op.get("curve_id") or op.get("base") or op.get("curve_kind")
    print(f"  [{i}] kind={op.get('kind')!r} ref={tag!r}")

res = apply_scenario_to_market(composed, market_json, AS_OF)
print("operations_applied:", res["operations_applied"], "warnings:", res["warnings"])

stressed = MarketContext.from_json(res["market_json"])
print("USD-OIS DF@1Y after:", stressed.get_discount("USD-OIS").df(1.0))
print("CORP-HAZARD survival@1Y after:", stressed.get_hazard("CORP-HAZARD").sp(1.0))
print("EUR/USD after stress:", stressed.fx.rate(Currency("EUR"), Currency("USD"), bd).rate)

def usd_ois_df_at_1y(mj: str) -> float:
    return MarketContext.from_json(mj).get_discount("USD-OIS").df(1.0)

def eurusd_rate(mj: str) -> float:
    context = MarketContext.from_json(mj)
    return context.fx.rate(Currency("EUR"), Currency("USD"), bd).rate

# Sequential application in priority order (5 → 10 → 20) to mirror typical composed ordering
seq_mj = market_json
for label, spec in (("FX +4%", fx), ("Rates +35bp", rates), ("Credit +60bp", credit)):
    step = apply_scenario_to_market(spec, seq_mj, AS_OF)
    seq_mj = step["market_json"]
    print(f"Sequential {label}: operations_applied={step['operations_applied']}")

print(
    "Sanity check — USD-OIS DF@1Y knot | composed vs sequential:",
    usd_ois_df_at_1y(res["market_json"]),
    usd_ois_df_at_1y(seq_mj),
)
print(
    "Sanity check — EUR/USD quote | composed vs sequential:",
    eurusd_rate(res["market_json"]),
    eurusd_rate(seq_mj),
)

# Second pass: apply another small parallel rate shock to already stressed market (sequential application)
follow = build_scenario_spec(
    "f2",
    ops_json(OperationSpec.curve_parallel_bp(DISCOUNT, "USD-OIS", 10.0)),
    "Follow-on rates +10bp",
    "",
)
res2 = apply_scenario_to_market(follow, res["market_json"], AS_OF)
print("Second apply operations_applied:", res2["operations_applied"])
d2 = MarketContext.from_json(res2["market_json"])
print("USD-OIS DF@1Y after 2nd shock:", d2.get_discount("USD-OIS").df(1.0))


Market with typed FX quote; EUR/USD=1.08
Composed spec priorities embedded; merged id: f+r+c
Merged operation stack:
  [0] kind='market_fx_pct' ref='EUR'
  [1] kind='curve_parallel_bp' ref='USD-OIS'
  [2] kind='curve_parallel_bp' ref='CORP-HAZARD'


operations_applied: 3 warnings: []
USD-OIS DF@1Y after: 0.9516633425566962
CORP-HAZARD survival@1Y after: 0.9711734837269662
EUR/USD after stress: 1.1232000000000002
Sequential FX +4%: operations_applied=1
Sequential Rates +35bp: operations_applied=1


Sequential Credit +60bp: operations_applied=1
Sanity check — USD-OIS DF@1Y knot | composed vs sequential: 0.9516633425566962 0.9516633425566962
Sanity check — EUR/USD quote | composed vs sequential: 1.1232000000000002 1.1232000000000002
Second apply operations_applied: 1
USD-OIS DF@1Y after 2nd shock: 0.9507121548872398


## Takeaways

- **`compose_scenarios`** merges multiple JSON specs—use it for **rates + credit + FX** (and more) in one application.
- Author each factor's operations with the typed **`OperationSpec`** builders (`curve_parallel_bp`, `market_fx_pct`); `build_scenario_spec` then wraps them with an id, labels, and a priority.
- **`priority`** on each spec controls merge ordering; print the composed JSON to confirm the final **`operations`** stack.
- **Historical templates** list via **`list_builtin_templates`**; drill into building blocks with **`list_template_components`**.
- **Chained application** (`apply` on an already stressed `market_json`) models **sequential** stress timelines; composition models **one combined** shock set.
- **FX shocks** need the pair present in the typed `FxMatrix` before the market is serialized — `_shared.build_demo_market` sets up EUR/USD for you.